# Run 25 (Kaggle, T4 ×2) — Plan 2: B2-scale EffNet @ 256×256 RGB

From-scratch EfficientNet-B2-scale (**7.83M**, 23 MBConv blocks) at **256×256 RGB**, compound-scaled
up from the Run 24 config that broke the plateau (66 → 70.45% val). Competition-legal: PyTorch
primitives only, no pretrained weights.

**Kaggle setup:**
- Add the **signal-object-detection** competition as an *Input* (right panel → Add Input).
- Accelerator: **GPU T4 ×2** — the notebook auto-uses both via `nn.DataParallel` and scales the batch.
- `submission.csv` is written to `/kaggle/working/` (Output tab / Submit).

Recipe: dropout 0.4, drop_path 0.2, translation aug, EMA 0.999, OneCycleLR, 70 epochs. `N_FOLDS=1` to
calibrate (confirm > 70.45), then 3 + TTA for the submission.

In [1]:
# Kaggle provides torch + the competition data — nothing to install.

In [2]:
# === Device (T4 x2) ===
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| device:', device, '| GPUs:', NGPU,
      '|', [torch.cuda.get_device_name(i) for i in range(NGPU)] if use_amp else 'CPU')

torch 2.10.0+cu128 | device: cuda | GPUs: 2 | ['Tesla T4', 'Tesla T4']


In [3]:
# === Locate the competition data under /kaggle/input (robust + diagnostic) ===
import glob
from pathlib import Path
print('Contents of /kaggle/input:')
for p in sorted(glob.glob('/kaggle/input/*')):
    print('  -', p)
    for q in sorted(glob.glob(p+'/*'))[:10]:
        print('       ', Path(q).name + ('/' if Path(q).is_dir() else ''))

hits = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True), key=len)
assert hits, ('No train.csv under /kaggle/input. Right panel -> "+ Add Input" -> Competitions tab -> '
              'search "signal object detection" -> add it, then re-run.')
DATA_DIR = Path(hits[0]).parent

def imgdir(stem):
    d = DATA_DIR/stem
    if d.is_dir(): return d
    cands=[Path(x) for x in glob.glob(str(DATA_DIR)+'/**/'+stem, recursive=True) if Path(x).is_dir()]
    return cands[0] if cands else d
TRAIN_DIR, TEST_DIR = imgdir('train'), imgdir('test')
print('\nDATA_DIR  =', DATA_DIR)
print('TRAIN_DIR =', TRAIN_DIR, '| #files:', len(glob.glob(str(TRAIN_DIR)+'/*')))
print('TEST_DIR  =', TEST_DIR,  '| #files:', len(glob.glob(str(TEST_DIR)+'/*')))
assert TRAIN_DIR.is_dir() and TEST_DIR.is_dir(), 'Found train.csv but not train/ & test/ folders — check the listing above and set TRAIN_DIR/TEST_DIR manually.'

Contents of /kaggle/input:
  - /kaggle/input/datasets
        robertnchis/

DATA_DIR  = /kaggle/input/datasets/robertnchis/my-database
TRAIN_DIR = /kaggle/input/datasets/robertnchis/my-database/train | #files: 15500
TEST_DIR  = /kaggle/input/datasets/robertnchis/my-database/test | #files: 5500


In [4]:
# === 4. Dataset (224x224 RGB, read on-the-fly) + augmentation ===
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

IMG = 256
train_df = pd.read_csv(DATA_DIR/'train.csv')
test_df  = pd.read_csv(DATA_DIR/'test.csv')

# per-channel mean/std from a sample (RGB spectrograms != ImageNet stats)
_smp = train_df['id'].sample(800, random_state=0)
_acc = np.zeros(3); _acc2 = np.zeros(3); _n = 0
for fn in _smp:
    a = np.asarray(Image.open(TRAIN_DIR/fn).convert('RGB').resize((IMG, IMG)), np.float32)/255.
    _acc += a.reshape(-1,3).sum(0); _acc2 += (a.reshape(-1,3)**2).sum(0); _n += IMG*IMG
MEAN = (_acc/_n); STD = np.sqrt(_acc2/_n - MEAN**2)
MEAN = torch.tensor(MEAN, dtype=torch.float32).view(3,1,1)
STD  = torch.tensor(STD,  dtype=torch.float32).view(3,1,1)
print('RGB mean', MEAN.flatten().tolist(), 'std', STD.flatten().tolist())

class SigDS(Dataset):
    def __init__(self, df, img_dir, train=True, fshift=18, tshift=28):
        self.df=df.reset_index(drop=True); self.dir=Path(img_dir); self.train=train
        self.fs=fshift; self.ts=tshift
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        a=np.asarray(Image.open(self.dir/row['id']).convert('RGB').resize((IMG,IMG)), np.float32)/255.
        x=torch.from_numpy(a).permute(2,0,1)              # [3,224,224]
        if self.train:
            fs=int(torch.randint(-self.fs,self.fs+1,(1,)).item())  # freq shift (rows)
            ts=int(torch.randint(-self.ts,self.ts+1,(1,)).item())  # time shift (cols)
            x=torch.roll(x,shifts=(fs,ts),dims=(1,2))
            if fs>0: x[:,:fs,:]=0
            elif fs<0: x[:,fs:,:]=0
            if ts>0: x[:,:,:ts]=0
            elif ts<0: x[:,:,ts:]=0
            c=0.8+0.4*torch.rand(1).item(); b=(torch.rand(1).item()-0.5)*0.1
            x=((x-0.5)*c+0.5+b).clamp_(0,1)
            x.add_(torch.randn_like(x)*0.02).clamp_(0,1)
        x=(x-MEAN)/STD
        label=int(row['label'])-1 if 'label' in row else -1
        return x, label
print('train', len(train_df), 'test', len(test_df))

RGB mean [0.2659030258655548, 0.051446329802274704, 0.3602016568183899] std [0.026629658415913582, 0.11764228343963623, 0.055583685636520386]
train 15500 test 5500


In [5]:
# === 5. Model — from-scratch EfficientNet/MBConv (PyTorch primitives only) ===
class DropPath(nn.Module):
    def __init__(s,p=0.0): super().__init__(); s.p=p
    def forward(s,x):
        if s.p==0.0 or not s.training: return x
        k=1-s.p; m=torch.empty((x.size(0),1,1,1),dtype=x.dtype,device=x.device).bernoulli_(k); return x/k*m

class MBConv(nn.Module):
    def __init__(s,cin,cout,k=3,st=1,t=6,se=0.25,dp=0.0):
        super().__init__()
        cm=cin*t; s.use_res=(st==1 and cin==cout); L=[]
        if t!=1: L+=[nn.Conv2d(cin,cm,1,bias=False),nn.BatchNorm2d(cm),nn.SiLU(True)]
        L+=[nn.Conv2d(cm,cm,k,st,k//2,groups=cm,bias=False),nn.BatchNorm2d(cm),nn.SiLU(True)]
        s.conv=nn.Sequential(*L); sc=max(1,int(cin*se))
        s.se=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(cm,sc,1),nn.SiLU(True),nn.Conv2d(sc,cm,1),nn.Sigmoid())
        s.proj=nn.Sequential(nn.Conv2d(cm,cout,1,bias=False),nn.BatchNorm2d(cout)); s.dp=DropPath(dp)
        if s.use_res: nn.init.zeros_(s.proj[1].weight)
    def forward(s,x):
        o=s.conv(x); o=o*s.se(o); o=s.proj(o); return x+s.dp(o) if s.use_res else o

class EffNet(nn.Module):
    # (expand t, out_ch, kernel, stride, repeats) — EfficientNet-B0 layout
    cfg=[(1,16,3,1,1),(6,24,3,2,2),(6,40,5,2,2),(6,80,3,2,3),(6,112,5,1,3),(6,192,5,2,4),(6,320,3,1,1)]
    def __init__(s,num_classes=5,dropout=0.3,drop_path=0.1,width=1.0,depth_mult=1.0):
        super().__init__()
        import math
        ch=lambda c:int(c*width)
        rep=lambda r:int(math.ceil(r*depth_mult))
        s.stem=nn.Sequential(nn.Conv2d(3,ch(32),3,2,1,bias=False),nn.BatchNorm2d(ch(32)),nn.SiLU(True))
        blocks=[]; cin=ch(32); tot=sum(rep(r) for *_,r in s.cfg); bi=0
        for t,co,k,st,r in s.cfg:
            co=ch(co)
            for j in range(rep(r)):
                blocks.append(MBConv(cin,co,k,st if j==0 else 1,t,dp=drop_path*bi/max(1,tot-1))); cin=co; bi+=1
        s.blocks=nn.Sequential(*blocks)
        s.head=nn.Sequential(nn.Conv2d(cin,ch(1280),1,bias=False),nn.BatchNorm2d(ch(1280)),nn.SiLU(True),
                             nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(ch(1280),num_classes))
        for m in s.modules():
            if isinstance(m,nn.Conv2d) and m.groups==1: nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m,nn.Linear): nn.init.zeros_(m.bias) if m.bias is not None else None
    def forward(s,x): return s.head(s.blocks(s.stem(x)))

_m=EffNet(width=1.1,depth_mult=1.2); _n=sum(p.numel() for p in _m.parameters()); print(f'EffNet B2-scale params: {_n:,} ({_n/1e6:.2f}M)')
print('out', tuple(_m(torch.randn(2,3,IMG,IMG)).shape))

EffNet B2-scale params: 7,825,533 (7.83M)
out (2, 5)


In [6]:
# === 6. EMA ===
import copy
class EMA:
    def __init__(s,m,d=0.999): s.d=d; s.sh=copy.deepcopy(m).eval(); [p.requires_grad_(False) for p in s.sh.parameters()]
    @torch.no_grad()
    def update(s,m):
        for a,b in zip(s.sh.state_dict().values(),m.state_dict().values()):
            if a.dtype.is_floating_point: a.mul_(s.d).add_(b,alpha=1-s.d)
            else: a.copy_(b)

In [7]:
CONFIG = dict(epochs=70, batch_size=48, max_lr=1.0e-3, weight_decay=1e-3,
              label_smoothing=0.1, dropout=0.4, drop_path=0.2, ema_decay=0.999,
              width=1.1, depth_mult=1.2,                # ~EfficientNet-B2 scale
              ema_reset_epoch=3, pct_start=0.2, num_workers=4)

from sklearn.model_selection import StratifiedKFold

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct=tot=0
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        for x,yb in loader:
            p=model(x.to(device)).float().argmax(1).cpu()
            correct+=(p==yb).sum().item(); tot+=yb.size(0)
    return 100*correct/tot

def train_fold(tr_df, va_df, cfg, tag='fold0', seed=42):
    torch.manual_seed(seed)
    bs = cfg['batch_size'] * max(1, NGPU)              # split across both T4s
    tl=DataLoader(SigDS(tr_df,TRAIN_DIR,True), batch_size=bs, shuffle=True,
                  num_workers=cfg['num_workers'], drop_last=True, persistent_workers=True, pin_memory=use_amp)
    vl=DataLoader(SigDS(va_df,TRAIN_DIR,False), batch_size=256, shuffle=False,
                  num_workers=cfg['num_workers'], persistent_workers=True, pin_memory=use_amp)
    print(f'[{tag}] {len(tr_df)} train / {len(va_df)} val | {len(tl)} batches/epoch | {NGPU} GPU(s), batch {bs} (first epoch slow — reading PNGs)',flush=True)
    core=EffNet(5,cfg['dropout'],cfg['drop_path'],width=cfg['width'],depth_mult=cfg['depth_mult']).to(device)
    model=nn.DataParallel(core) if NGPU>1 else core    # use both GPUs for fwd/bwd
    ema=EMA(core,cfg['ema_decay'])                      # EMA tracks the (shared) core weights
    crit=nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['max_lr'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
    steps=len(tl); sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=cfg['max_lr'],
              total_steps=cfg['epochs']*steps,pct_start=cfg['pct_start'])
    best=0.0; best_state=None
    for ep in range(cfg['epochs']):
        if ep==cfg['ema_reset_epoch']: ema=EMA(core,cfg['ema_decay'])
        model.train(); tc=torch.zeros((),device=device); tn=0
        for i,(x,yb) in enumerate(tl):
            x,yb=x.to(device,non_blocking=True),yb.to(device,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
                out=model(x); loss=crit(out,yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(core)
            with torch.no_grad(): tc+=(out.float().argmax(1)==yb).sum()
            tn+=yb.size(0)
            if (i+1)%50==0: print(f'  [{tag}] ep{ep+1} batch {i+1}/{len(tl)}',flush=True)
        ta=100*tc.item()/tn; rv=evaluate(core,vl); ev=evaluate(ema.sh,vl)
        acc=max(rv,ev); s='EMA' if ev>=rv else 'raw'
        print(f'[{tag}] ep{ep+1}/{cfg["epochs"]} train {ta:.2f} raw {rv:.2f} ema {ev:.2f} -> val {acc:.2f} ({s}) gap {ta-acc:+.1f} lr {opt.param_groups[0]["lr"]:.2e}',flush=True)
        if acc>best:
            best=acc; chosen=ema.sh if ev>=rv else core
            best_state={k:v.detach().cpu().clone() for k,v in chosen.state_dict().items()}
            torch.save(best_state, f'/kaggle/working/best_{tag}.pt')
    print(f'[{tag}] BEST {best:.2f}%'); return best, best_state

In [ ]:
  # === Pre-cache all images in RAM (avoids repeated disk reads) ===
  import numpy as np, torch, torch.nn.functional as F
  from PIL import Image
  from pathlib import Path

  def cache_split(df, img_dir):
      X = torch.empty(len(df), 3, IMG, IMG, dtype=torch.float32)
      for i, row in enumerate(df.itertuples(index=False)):
          a = np.asarray(Image.open(img_dir/row.id).convert('RGB').resize((IMG,IMG)), np.float32)/255.
          X[i] = torch.from_numpy(a).permute(2,0,1)
          if (i+1)%3000==0: print(f'  cached {i+1}/{len(df)}', flush=True)
      return X

  print('Caching train images...'); Xtr_cache = cache_split(train_df, DATA_DIR/'train')
  print('Caching test images...');  Xte_cache = cache_split(test_df,  DATA_DIR/'test')
  print(f'RAM used: train {Xtr_cache.nbytes/1e9:.1f}GB  test {Xte_cache.nbytes/1e9:.1f}GB')


Caching train images...
  cached 3000/15500
  cached 6000/15500
  cached 9000/15500


In [ ]:
# === 8. Run K-fold ===
N_FOLDS = 1   # calibrate first; set 3 for the submission
skf=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
splits=list(skf.split(train_df,train_df['label']))[:N_FOLDS]
fold_states=[]; accs=[]
for f,(tri,vai) in enumerate(splits):
    print(f'\n===== FOLD {f+1}/{N_FOLDS} =====',flush=True)
    b,st=train_fold(train_df.iloc[tri], train_df.iloc[vai], CONFIG, tag=f'fold{f}', seed=42+f)
    accs.append(b); fold_states.append(st)
print(f'\nCV mean {np.mean(accs):.2f}%  (Kaggle est ~{np.mean(accs)+2.7:.1f}%)')

In [ ]:
# === 9. Ensemble + TTA inference ===
import torch.nn.functional as F
def tta(x):
    yield x
    yield (((x*STD.to(x.device)+MEAN.to(x.device)-0.5)*0.9+0.5).clamp(0,1)-MEAN.to(x.device))/STD.to(x.device)
test_loader=DataLoader(SigDS(test_df,TEST_DIR,False), batch_size=128, shuffle=False, num_workers=CONFIG['num_workers'])
probs=torch.zeros(len(test_df),5); ids=test_df['id'].tolist()
for st in fold_states:
    model=EffNet(5,CONFIG['dropout'],CONFIG['drop_path'],width=CONFIG['width'],depth_mult=CONFIG['depth_mult']).to(device); model.load_state_dict(st); model.eval()
    off=0
    with torch.no_grad(), torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
        for x,_ in test_loader:
            x=x.to(device); acc=torch.zeros(x.size(0),5)
            for v in tta(x): acc+=F.softmax(model(v).float(),1).cpu()
            probs[off:off+x.size(0)]+=acc/2; off+=x.size(0)
pred=probs.argmax(1).numpy()+1
sub=pd.DataFrame({'id':ids,'label':pred}); sub.to_csv('/kaggle/working/submission.csv',index=False)
print(sub['label'].value_counts().sort_index()); print(sub.head())
print('submission.csv -> /kaggle/working/submission.csv')